# Exploratory Data Analysis
Account Takeover (ATO) detection dataset.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.loader import load

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 100

SAMPLE_SIZE = 100_000
RANDOM_STATE = 42
TARGET_COL = "Is Account Takeover"

## 1. Overview
Load dataset, inspect shape, column types, and sample rows.

In [ ]:
df = load()

print(f"Shape: {df.shape[0]:,} rows  x  {df.shape[1]} columns")
print()
print("Column names and dtypes:")
print(df.dtypes.to_string())
print()
print("First 5 rows:")
df.head()

## 2. Target Analysis
Classification target: `Is Account Takeover`. Inspect class distribution.

In [ ]:
counts = df[TARGET_COL].value_counts().sort_index()
labels = {False: "Legitimate (False)", True: "Account Takeover (True)"}
colors = ["#4C72B0", "#DD8452"]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(
    [labels[k] for k in counts.index],
    counts.values,
    color=colors,
    edgecolor="white",
    width=0.5,
)
for bar, val in zip(bars, counts.values):
    pct = val / len(df) * 100
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() * 1.05,
        f"{val:,}\n({pct:.4f}%)",
        ha="center", va="bottom", fontsize=11, fontweight="bold",
    )
ax.set_yscale("log")
ax.set_title("Target Class Distribution: Is Account Takeover", fontsize=14, pad=12)
ax.set_ylabel("Count (log scale)")
ax.set_xlabel("Class")
plt.tight_layout()
plt.show()

print(f"Class balance:\n{counts.to_string()}")
print(f"\nImbalance ratio: {counts[False] / counts[True]:.0f}:1")

## 3. Missing Values
Heatmap of null percentage per column.

In [ ]:
null_pct = (df.isnull().mean() * 100).to_frame("Null %").T

fig, ax = plt.subplots(figsize=(14, 3))
sns.heatmap(
    null_pct,
    annot=True,
    fmt=".1f",
    cmap="YlOrRd",
    vmin=0,
    vmax=100,
    ax=ax,
    linewidths=0.5,
    cbar_kws={"label": "Null %", "shrink": 0.8},
)
ax.set_title("Missing Values by Column (%)", fontsize=14)
ax.set_xlabel("Column")
ax.set_yticklabels([])
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

print("Null counts:")
print(df.isnull().sum()[df.isnull().sum() > 0].to_string())

## 4. Feature Distributions
3x3 grid of histograms for numeric and boolean features (sampled).

In [ ]:
sample = df.sample(min(SAMPLE_SIZE, len(df)), random_state=RANDOM_STATE)

numeric_cols = df.select_dtypes("number").columns.tolist()
bool_cols = [c for c in df.select_dtypes("bool").columns if c != TARGET_COL]
plot_cols = numeric_cols + bool_cols  # up to 6 columns

n_cols = 3
n_rows = 3
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 12))
axes_flat = axes.flatten()

for i, col in enumerate(plot_cols[:9]):
    ax = axes_flat[i]
    data = sample[col].dropna().astype(float)
    ax.hist(data, bins=40, color="#4C72B0", edgecolor="white", alpha=0.85)
    ax.set_title(col, fontsize=11, fontweight="bold")
    ax.set_xlabel(col)
    ax.set_ylabel("Count")

for j in range(len(plot_cols), len(axes_flat)):
    axes_flat[j].set_visible(False)

fig.suptitle(f"Feature Distributions  (sample n={min(SAMPLE_SIZE, len(df)):,})", fontsize=14)
plt.tight_layout()
plt.show()

## 5. Correlation Matrix
Pearson correlation between numeric features (annotated heatmap).

In [ ]:
corr = sample[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    ax=ax,
    square=True,
    linewidths=0.5,
    cbar_kws={"shrink": 0.8},
)
ax.set_title("Correlation Matrix — Numeric Features", fontsize=14, pad=12)
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

## 6. Features vs Target
Box plots of the three strongest features split by `Is Account Takeover`.
Minority class (141 rows) is combined with an equal-size majority sample for visibility.

In [ ]:
# Oversample minority to make box plots readable
pos = df[df[TARGET_COL] == True]
neg = df[df[TARGET_COL] == False].sample(len(pos) * 500, random_state=RANDOM_STATE)
plot_df = pd.concat([pos, neg]).copy()

feature_cols = ["Round-Trip Time [ms]", "ASN", "Login Successful"]

fig, axes = plt.subplots(1, 3, figsize=(15, 6))
palette = {False: "#4C72B0", True: "#DD8452"}

for ax, col in zip(axes, feature_cols):
    plot_df[col] = plot_df[col].astype(float)
    sns.boxplot(
        data=plot_df,
        x=TARGET_COL,
        y=col,
        palette=palette,
        width=0.5,
        flierprops={"marker": ".", "alpha": 0.3},
        ax=ax,
    )
    ax.set_title(f"{col}\nvs Is Account Takeover", fontsize=11, fontweight="bold")
    ax.set_xlabel("Is Account Takeover")
    ax.set_ylabel(col)

fig.suptitle("Feature Distributions by Target Class", fontsize=14)
plt.tight_layout()
plt.show()

## 7. Key Findings

- **Extreme class imbalance**: Only 141 ATO events in 31.3 million logins (~0.00045%). Any model must use class weighting, SMOTE, or anomaly-detection framing — accuracy is a meaningless metric here.
- **Round-Trip Time is 95.9% missing**: RTT was only captured for a small subset of logins, making it unreliable as a standalone feature. Consider using it as a binary indicator ("RTT was recorded") rather than its raw value.
- **All 141 ATO events are `Is Attack IP = True`**: The `Is Attack IP` flag is a near-perfect predictor in this dataset. In production, investigate whether this flag is available at inference time or is itself a post-hoc label — if post-hoc, it must be excluded from training features.
- **User IDs span the full int64 range**: IDs appear to be hashed or randomly assigned rather than sequential, so their raw numeric value carries no ordinal signal. Encode them categorically or use frequency/target encoding.
- **Geographic features (Country, Region, City) are nearly complete** (<0.2% null) and likely carry strong signal — logins from unusual countries or regions for a given user are a classic ATO indicator worth engineering as a behavioural feature.